# Served demand (Huff), with imputed competitors

**Question:** of each cell's addressable women, how many does each premium supplier (our lounges and the premium salons) serve, and how many does nobody serve?

**The model, as agreed on 2026-10-11** (one model for lounges and growth):
1. **Demand = served women** (`src/model/served.py`). Each cell's addressable women are split, Huff-style, across every premium salon and every lounge within 20 minutes of it, across emirate borders. Weight = reviews^ALPHA × (rating/4.6)^RATING_ELASTICITY × DECAY[drive-time band], plus an outside option (women who go elsewhere or use no premium salon). Drive-time bands come from the cell's own isochrones, else from straight-line bands fitted on those. Cells our salon search never covered get an imputed stand-in competitor (emirate × density tier).
2. **Cost = site cost** (`src/features/lounges.py`): shop rent × 150 m² + an assumed fit-out written off over 5 years. Cost efficiency = served women per AED 1k of site cost; `cost_index` = that ÷ the emirate's median lounge (1 = typical).
3. **Lounge scorecard** = served demand + cost index. Cannibalisation (the share of a lounge's women its siblings would absorb if it closed) is shown, not scored.
4. **Growth** = what-if: open a typical lounge (median lounge reviews and rating) at each area's centroid, and count the women it serves, how many it takes from our lounges, and the net new.

**Every parameter here is a first cut**, not calibrated. This notebook shows what they produce so we can decide together.

In [1]:
import sys, pathlib, statistics
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 120)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
from src.features import lounges as L
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))
F = pd.DataFrame([x.model_dump() for x in r.features]).set_index("branch_id")
D = {d.branch_id: d for d in r.decisions}
AR = pd.DataFrame([x.model_dump(exclude={"cell_ids"}) for x in r.areas]).set_index("area_id")
AR["call"] = AR.index.map({d.area_id: d.action for d in r.area_decisions})

## 1. Parameters and reach

In [2]:
acc = S._ACCESS[id(v3)]
print("ALPHA", S.ALPHA, "| RATING_ELASTICITY", S.RATING_ELASTICITY, "| DECAY", S.DECAY, "| OUTSIDE_PCTL", S.OUTSIDE_PCTL)
print("cell x supplier pairs within 20 min:", len(acc[1]), acc[1].source.value_counts().to_dict())
print("straight-line band edges (km), the 75th pct of isochrone-reached pairs:", {k: round(x, 1) for k, x in acc[2].items()})

ALPHA 1.0 | RATING_ELASTICITY 5.0 | DECAY {10: 1.0, 15: 0.5, 20: 0.25} | OUTSIDE_PCTL 50
cell x supplier pairs within 20 min: 138776 {'isochrone': 97650, 'straight_line': 41126}
straight-line band edges (km), the 75th pct of isochrone-reached pairs: {10: 5.0, 15: 8.7, 20: 12.8}


## 2. Where the UAE's addressable women go

In [3]:
sv = L._SERVED[(id(v3), r.levels.worker_share, r.levels.affluence, frozenset())]
cells = sv["cells"].join(v3.cells[["emirate"]])
t = cells.groupby("emirate")[["demand", "served_lounges", "served_competitors", "unserved"]].sum()
t.loc["UAE"] = t.sum()
(t.div(t.demand, axis=0)[["served_lounges", "served_competitors", "unserved"]] * 100).round(1).assign(demand=t.demand.round(-3))

,served_lounges,served_competitors,unserved,demand
emirate,,,,
Abu Dhabi,13.1,74.9,12.0,954000.0
Ajman,1.3,87.4,11.2,154000.0
Dubai,1.5,97.3,1.2,1000000.0
Fujairah,3.6,75.2,21.3,92000.0
Ras Al Khaimah,5.1,83.3,11.6,145000.0
Sharjah,2.1,84.8,13.1,600000.0
Umm Al Quwain,0.0,74.5,25.5,34000.0
UAE,5.5,85.5,9.0,2978000.0


## 3. Per lounge: served demand, absorption, and v3's estimate

In [4]:
f = F[~F.not_scored].rename(columns={"review_count": "reviews", "served_women": "served", "absorbed_share": "absorbed",
                                     "est_customers": "v3_est_customers", "addressable_women": "catchment_addressable"})
f = f[["emirate", "reviews", "rating", "served", "absorbed", "v3_est_customers", "catchment_addressable"]]
print("Spearman served vs reviews:", round(f.served.corr(f.reviews, method="spearman"), 2),
      "| served vs v3 est_customers:", round(f.served.corr(f.v3_est_customers, method="spearman"), 2))
f.sort_values("served", ascending=False).round(2)

Spearman served vs reviews: 0.4 | served vs v3 est_customers: 0.9


,emirate,reviews,rating,served,absorbed,v3_est_customers,catchment_addressable
branch_id,,,,,,,
al-ain,Abu Dhabi,1332,4.7,18651.84,0.00,12617.50,102438.95
al-dhafra,Abu Dhabi,359,4.5,17633.50,0.00,16246.70,22927.30
khalifa-city-a,Abu Dhabi,2465,4.5,16679.26,0.20,11486.57,91431.59
shakhbout-city,Abu Dhabi,1277,4.5,12743.27,0.19,12239.50,134563.91
baniyas,Abu Dhabi,697,4.5,11086.02,0.13,9999.47,106825.82
shahama,Abu Dhabi,757,4.6,9865.31,0.19,5725.09,65769.16
al-falah,Abu Dhabi,951,4.6,9509.65,0.15,27054.29,49946.83
ministries-complex,Abu Dhabi,1217,4.9,8397.89,0.16,4882.56,107721.68
ras-al-khaimah,Ras Al Khaimah,772,4.5,7328.28,0.00,7713.24,55839.70


## 4. Sensitivity: how the parameters move served demand

In [5]:
from src.market import women_15plus, affluence_weight
w = women_15plus(v3.cells, v3.emirates, 0.055); aw = w * affluence_weight(v3.cells.rent_observed, w, 0.5)
ns = frozenset(L.NOT_SCORED); base = (S.ALPHA, dict(S.DECAY), S.OUTSIDE_PCTL); rows = []
for alpha in (0.5, 1.0):
    for pct in (25, 50, 75):
        for decay in ({10: 1, 15: 0.5, 20: 0.25}, {10: 1, 15: 0.7, 20: 0.5}):
            S.ALPHA, S.DECAY, S.OUTSIDE_PCTL = alpha, decay, pct
            o = S.served(v3, a, aw, not_scored=ns); c = o["cells"]; s = pd.Series(o["served"])
            rows.append(dict(alpha=alpha, outside_pct=pct, decay15_20=f"{decay[15]}/{decay[20]}",
                             unserved_pct=round(100 * c.unserved.sum() / c.demand.sum(), 1),
                             lounges_pct=round(100 * c.served_lounges.sum() / c.demand.sum(), 1),
                             rank_corr_vs_base=round(s.corr(f.served, method="spearman"), 2),
                             top3=", ".join(s.sort_values(ascending=False).index[:3])))
S.ALPHA, S.DECAY, S.OUTSIDE_PCTL = base
pd.DataFrame(rows)

,alpha,outside_pct,decay15_20,unserved_pct,lounges_pct,rank_corr_vs_base,top3
0,0.5,25,0.5/0.25,2.4,3.7,0.95,"al-dhafra, al-ain, baniyas"
1,0.5,25,0.7/0.5,1.8,3.7,0.95,"al-dhafra, al-ain, khalifa-city-a"
2,0.5,50,0.5/0.25,8.1,3.4,0.95,"al-dhafra, al-ain, baniyas"
3,0.5,50,0.7/0.5,6.5,3.5,0.96,"al-dhafra, al-ain, khalifa-city-a"
4,0.5,75,0.5/0.25,10.7,3.3,0.96,"al-dhafra, al-ain, shakhbout-city"
5,0.5,75,0.7/0.5,8.6,3.4,0.96,"al-dhafra, al-ain, khalifa-city-a"
6,1.0,25,0.5/0.25,1.0,5.9,1.00,"al-dhafra, al-ain, khalifa-city-a"
7,1.0,25,0.7/0.5,0.8,5.9,1.00,"al-dhafra, al-ain, khalifa-city-a"
8,1.0,50,0.5/0.25,9.0,5.5,1.00,"al-ain, al-dhafra, khalifa-city-a"
9,1.0,50,0.7/0.5,7.4,5.6,1.00,"al-ain, al-dhafra, khalifa-city-a"


## 5. Imputed competitors for unsearched cells

`IMPUTE_WHY`: printed below. How many stand-ins, where, and how big, against the real premium suppliers we found.

In [6]:
print(S.IMPUTE_WHY, "\n")
imp = S._IMPUTED[id(v3)][1].copy()
imp["cell_id"] = imp.id.str[4:]
imp = imp.join(v3.cells[["emirate", "name"]], on="cell_id").join(w.rename("women"), on="cell_id")
sup = S._suppliers(v3, a, not_scored=ns); real = sup[~sup.lounge & ~sup.id.str.startswith("IMP:")]
c = v3.cells; lat, lng = real.lat.to_numpy()[:, None], real.lng.to_numpy()[:, None]
inbox = (c.south.to_numpy() <= lat) & (lat < c.north.to_numpy()) & (c.west.to_numpy() <= lng) & (lng < c.east.to_numpy())
real = real.assign(emirate=np.where(inbox.any(1), c.emirate.to_numpy()[inbox.argmax(1)], "(outside grid)"))
print("imputed stand-ins:", len(imp), "| real premium salons:", len(real), "| lounges:", int(sup.lounge.sum()))
real_and_lounges = sup[~sup.id.str.startswith("IMP:")]
print("outside option (median supplier attractiveness): with stand-ins", np.percentile(sup.reviews.clip(lower=1) ** S.ALPHA, S.OUTSIDE_PCTL).round(1),
      "| without", np.percentile(real_and_lounges.reviews.clip(lower=1) ** S.ALPHA, S.OUTSIDE_PCTL).round(1))
t = imp.groupby("emirate").agg(stand_ins=("id", "size"), women_in_those_cells=("women", "sum"),
                               reviews_total=("reviews", "sum"), reviews_median=("reviews", "median"), reviews_max=("reviews", "max"))
t["implied_reviews_per_1k"] = t.reviews_total / (t.women_in_those_cells / 1000)
t = t.join(real.groupby("emirate").agg(real_premium=("id", "size"), real_reviews_total=("reviews", "sum")), how="outer")
t["stand_in_share_of_reviews"] = t.reviews_total / (t.reviews_total.fillna(0) + t.real_reviews_total)
t.round(2)

Cells our salon search never covered (remote or thinly searched) have no known competitors, so a new lounge there would look like it takes everything. Each such cell gets one stand-in competitor whose reviews = the premium reviews per 1k women of SEARCHED cells in the same emirate and density tier (tertile of women per cell; total reviews / total women, since most single cells hold none), x the cell's women. Assumption, 2026-10-11: supply is like that of comparable searched places in the same emirate. 

imputed stand-ins: 952 | real premium salons: 1129 | lounges: 23
outside option (median supplier attractiveness): with stand-ins 113.0 | without 201.5


,stand_ins,women_in_those_cells,reviews_total,reviews_median,reviews_max,implied_reviews_per_1k,real_premium,real_reviews_total,stand_in_share_of_reviews
emirate,,,,,,,,,
(outside grid),NaN,NaN,NaN,NaN,NaN,NaN,6,1633.0,NaN
Abu Dhabi,680.0,295934.70,17021.62,3.36,275.73,57.52,304,86908.0,0.16
Ajman,5.0,3702.50,96.84,19.79,40.08,26.16,38,11073.0,0.01
Dubai,145.0,72027.37,25125.35,160.91,340.29,348.83,582,233248.0,0.10
Fujairah,12.0,14175.30,1209.08,97.07,164.23,85.29,18,3884.0,0.24
Ras Al Khaimah,57.0,38089.04,2134.84,20.56,146.46,56.05,33,6562.0,0.25
Sharjah,48.0,46734.57,796.73,5.62,147.04,17.05,143,34631.0,0.02
Umm Al Quwain,5.0,5402.03,160.80,30.94,39.84,29.77,5,1224.0,0.12


In [7]:
print("biggest stand-ins (reviews = the cell's women x its emirate/tier rate):")
imp.sort_values("reviews", ascending=False).head(10)[["cell_id", "name", "emirate", "women", "reviews"]].round(0)

biggest stand-ins (reviews = the cell's women x its emirate/tier rate):


,cell_id,name,emirate,women,reviews
184,r55c195,Al Marmoom,Dubai,598.0,340.0
108,r46c200,Dubai,Dubai,592.0,337.0
222,r58c201,Grayteesah,Dubai,591.0,337.0
199,r57c173,Hessyan 2,Dubai,584.0,332.0
129,r50c206,Al Meryal,Dubai,582.0,332.0
220,r58c197,Al Lesaily,Dubai,579.0,330.0
161,r53c178,Jebel Ali Port,Dubai,578.0,329.0
258,r61c178,Dubai Industrial City,Dubai,575.0,328.0
165,r53c198,Al Marmoom,Dubai,570.0,325.0
103,r45c206,Dubai,Dubai,566.0,322.0


### Effect: served demand with vs without imputation

Toggle by swapping `served._imputed` for an empty frame and clearing the reach and what-if caches; restored afterwards. The what-if also runs for the five areas with the largest net new (see `growth.ipynb`).

In [8]:
def snapshot():
    o = S.served(v3, a, aw, not_scored=ns); c = o["cells"].join(v3.cells[["emirate"]])
    g = c.groupby("emirate")[["demand", "served_lounges", "served_competitors", "unserved"]].sum(); g.loc["UAE"] = g.sum()
    scored = F[~F.not_scored]
    rev, rat = statistics.median(scored.review_count), statistics.median(scored.rating.dropna())
    top = AR.sort_values("new_site_net", ascending=False).head(5)
    wi = {i: S.what_if_open(v3, a, aw, x.lat, x.lng, rev, rat, not_scored=ns) for i, x in top.iterrows()}
    return g, pd.Series(o["served"]), wi

with_imp = snapshot()
orig = S._imputed
S._imputed = lambda v3, prem: pd.DataFrame(columns=["id", "lat", "lng", "reviews", "rating", "lounge"])
S._ACCESS.clear(); S._IMPUTED.clear(); L._SERVED.clear()
try:
    without = snapshot()
finally:
    S._imputed = orig; S._ACCESS.clear(); S._IMPUTED.clear(); L._SERVED.clear()

g1, g0 = with_imp[0], without[0]
pd.DataFrame({"unserved_pct_with": 100 * g1.unserved / g1.demand, "unserved_pct_without": 100 * g0.unserved / g0.demand,
              "lounges_pct_with": 100 * g1.served_lounges / g1.demand, "lounges_pct_without": 100 * g0.served_lounges / g0.demand}).round(1)

,unserved_pct_with,unserved_pct_without,lounges_pct_with,lounges_pct_without
emirate,,,,
Abu Dhabi,12.0,33.6,13.1,13.4
Ajman,11.2,18.9,1.3,1.3
Dubai,1.2,10.7,1.5,1.5
Fujairah,21.3,50.8,3.6,3.5
Ras Al Khaimah,11.6,33.0,5.1,5.1
Sharjah,13.1,26.6,2.1,2.1
Umm Al Quwain,25.5,39.5,0.0,0.0
UAE,9.0,24.3,5.5,5.6


In [9]:
lo = pd.DataFrame({"served_with": with_imp[1], "served_without": without[1]})
lo["change_pct"] = 100 * (lo.served_with / lo.served_without - 1)
print("lounge served: total with", round(lo.served_with.sum()), "| without", round(lo.served_without.sum()),
      "| Spearman", round(lo.served_with.corr(lo.served_without, method="spearman"), 3))
display(lo.sort_values("change_pct").round(1).head(8))
pd.DataFrame({i: {"net_new_with": with_imp[2][i]["net_new"], "net_new_without": without[2][i]["net_new"]} for i in with_imp[2]}).T.round(0)

lounge served: total with 164861 | without 167764 | Spearman 0.994


,served_with,served_without,change_pct
shahama,9865.3,13223.7,-25.4
al-falah,9509.6,11036.4,-13.8
noya-plaza,1385.7,1485.6,-6.7
westyas,3160.1,3293.0,-4.0
khalifa-city-a,16679.3,17349.7,-3.9
al-ain,18651.8,19203.8,-2.9
jumeirah-park,1701.9,1739.7,-2.2
mirdif-35,5489.5,5586.3,-1.7


,net_new_with,net_new_without
al-dhaid-sharjah,57932.0,59012.0
manama-ajman,35094.0,36425.0
khor-fakkan-sharjah,23424.0,24140.0
al-madam-sharjah,19728.0,22458.0
mudayrah-ajman,18651.0,22646.0


## Open questions for review

1. **The stand-ins also weaken the outside option** (section 5). The outside option is the median supplier's attractiveness, and the 952 stand-ins are counted in that median: it drops from 201.5 to 113.0. So the with/without comparison mixes two effects: UAE unserved demand is 9.0% with imputation and 24.3% without, and part of that gap is the weaker outside option, not the stand-ins themselves. Should the outside option be set on real suppliers only?
2. **Imputation barely changes the growth what-ifs.** Net new at al-dhaid is 57,932 with stand-ins and 59,012 without; mudayrah 18,651 vs 22,646. It does not fix the remote-area pull (top question in `growth.ipynb`).
3. **Stand-ins differ a lot by emirate.** Abu Dhabi gets 680 tiny ones (median 3.36 reviews); Dubai gets 145 large ones (median 160.91, up to 340.29), some in cells named Jebel Ali Port and Dubai Industrial City. Stand-ins hold 16% of Abu Dhabi's premium reviews and about a quarter in Fujairah and Ras Al Khaimah. Is "same emirate × density tier" a fair stand-in, and should non-residential cells get one?
4. **Unserved demand depends mainly on the outside option** (section 4): 0.8%–14.0% over the 12 combinations. Lounge rankings barely move (Spearman ≥ 0.95 against the base). What should the outside option mean: no premium salon at all, or willing to travel beyond 20 minutes?
5. **Dubai is harsh on our lounges.** Competitors serve 97.3% of Dubai's women and our lounges 1.5%. Four of the five Dubai lounges serve 1.7k–2.4k women (mirdif-35: 5.5k). Plausible, or should attractiveness be capped?
6. **No capacity cap yet:** al-ain serves 18,652 women.